In [1]:
import numpy as np
import torch
import os
import pandas as pd
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.utils import homophily, to_networkx, degree
import networkx as nx

/home/hice1/pmaji3/scratch/gnn02/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def calculate_stats(npz_path, dataset_name):
    # Load the NPZ file
    with np.load(npz_path, allow_pickle=True) as f:
        x = torch.from_numpy(f['node_features']).float()
        y = torch.from_numpy(f['node_labels']).long()
        edge_index = torch.from_numpy(f['edges']).long()
    
    if edge_index.shape[1] == 2 and edge_index.shape[0] != 2:
        edge_index = edge_index.t().contiguous()

    data = Data(x=x, y=y, edge_index=edge_index)

    edge_index, _ = torch_geometric.utils.remove_self_loops(edge_index)
    # Make undirected and remove duplicate edges (A->B and B->A count as 1)
    edge_index = torch_geometric.utils.to_undirected(edge_index)
    edge_index = torch_geometric.utils.coalesce(edge_index)
    num_unique_edges = edge_index.shape[1] // 2
    
    # 1. Basic Counts
    nodes = data.num_nodes
    edges = data.num_edges
    avg_degree = (2 * edges) / nodes
    classes = len(torch.unique(y))
    
    # 2. Homophily Metrics
    edge_homo = homophily(data.edge_index, data.y, method='edge')
    
    # Adjusted Homophily (Cohen's Kappa style)
    deg = degree(data.edge_index[0], nodes)
    deg_sum = torch.zeros(classes)
    deg_sum.index_add_(0, y, deg)
    expected_homo = torch.sum(deg_sum**2) / (edges**2)
    adj_homo = (edge_homo - expected_homo) / (1 - expected_homo)

    # 3. Clustering (Using NetworkX)
    G = to_networkx(data, to_undirected=True)
    global_clustering = nx.transitivity(G)
    avg_local_clustering = nx.average_clustering(G)
    
    # 4. Diameter (Warning: Very slow for large graphs like Cora/PubMed)
    # If the graph is disconnected, we take the diameter of the largest component
    if nodes < 5000: # Skip for very large graphs to save time
        largest_cc = max(nx.connected_components(G), key=len)
        diameter = nx.diameter(G.subgraph(largest_cc))
    else:
        #diameter = nx.diameter(G) 
        diameter="Too Large"

    return {
        "dataset": dataset_name,
        "nodes": nodes,
        "edges": edges,
        "unique_edges":num_unique_edges,
        "avg_degree": round(avg_degree, 2),
        "global_clustering": round(global_clustering, 3),
        "avg_local_clustering": round(avg_local_clustering, 3),
        "diameter": diameter,
        "node_features": x.shape[1],
        "classes": classes,
        "edge_homophily": round(float(edge_homo), 3),
        "adjusted_homophily": round(float(adj_homo), 3)
    }

In [19]:
import numpy as np
f = np.load("../tmp/amazon-fasttext-custom/amazon-fasttext/raw/amazon-fasttext.npz")
print(f.files)

['node_features', 'node_labels', 'edges', 'train_masks', 'val_masks', 'test_masks']


In [4]:
npz_path= "../tmp/amazon-fasttext-custom/amazon-fasttext/raw/amazon-fasttext.npz"
dataset_name ='amazon'
calculate_stats(npz_path, dataset_name)

{'dataset': 'amazon',
 'nodes': 24492,
 'edges': 113276,
 'unique_edges': 93050,
 'avg_degree': 9.25,
 'global_clustering': 0.316,
 'avg_local_clustering': 0.582,
 'diameter': 'Too Large',
 'node_features': 300,
 'classes': 5,
 'edge_homophily': 0.382,
 'adjusted_homophily': 0.153}

In [5]:
npz_path= "../tmp/Amazon-ratings/amazon_ratings/raw/amazon_ratings.npz"

dataset_name ='amazon'
calculate_stats(npz_path, dataset_name)

{'dataset': 'amazon',
 'nodes': 24492,
 'edges': 93050,
 'unique_edges': 93050,
 'avg_degree': 7.6,
 'global_clustering': 0.316,
 'avg_local_clustering': 0.582,
 'diameter': 'Too Large',
 'node_features': 300,
 'classes': 5,
 'edge_homophily': 0.38,
 'adjusted_homophily': 0.14}

In [6]:
npz_path= "../tmp/roman-fasttext-custom/roman-fasttext/raw/roman-fasttext.npz"


dataset_name ='Roman Empire'
calculate_stats(npz_path, dataset_name)

{'dataset': 'Roman Empire',
 'nodes': 22600,
 'edges': 32684,
 'unique_edges': 32684,
 'avg_degree': 2.89,
 'global_clustering': 0.291,
 'avg_local_clustering': 0.388,
 'diameter': 'Too Large',
 'node_features': 300,
 'classes': 18,
 'edge_homophily': 0.043,
 'adjusted_homophily': -0.055}

In [ ]:
npz_path= "../tmp/roman-fasttext-custom/roman-fasttext/raw/roman-fasttext.npz"


dataset_name ='Roman Empire'
calculate_stats(npz_path, dataset_name)

In [10]:
import torch
import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx, homophily, degree
import networkx as nx

def calculate_stats_from_data(data, dataset_name):
    edge_index = data.edge_index
    
    # Pre-processing same as your original script
    edge_index, _ = torch_geometric.utils.remove_self_loops(edge_index)
    edge_index = torch_geometric.utils.to_undirected(edge_index)
    edge_index = torch_geometric.utils.coalesce(edge_index)
    
    nodes = data.num_nodes
    edges = edge_index.shape[1] // 2 # Count unique undirected edges
    avg_degree = (2 * edges) / nodes
    classes = len(torch.unique(data.y))

    # Homophily
    edge_homo = homophily(edge_index, data.y, method='edge')
    
    # Adjusted Homophily
    deg = degree(edge_index[0], nodes)
    deg_sum = torch.zeros(classes)
    deg_sum.index_add_(0, data.y, deg)
    # Total volume is sum of degrees (2 * edges)
    expected_homo = torch.sum(deg_sum**2) / ((2 * edges)**2)
    adj_homo = (edge_homo - expected_homo) / (1 - expected_homo)

    # Graph Metrics
    G = to_networkx(data, to_undirected=True)
    global_clustering = nx.transitivity(G)
    avg_local_clustering = nx.average_clustering(G)
    
    # Cora is small enough for diameter (~2708 nodes)
    largest_cc = max(nx.connected_components(G), key=len)
    diameter = nx.diameter(G.subgraph(largest_cc))

    return {
        "dataset": dataset_name,
        "nodes": nodes,
        "edges": edges,
        "avg_degree": round(avg_degree, 2),
        "global_clustering": round(global_clustering, 3),
        "avg_local_clustering": round(avg_local_clustering, 3),
        "diameter": diameter,
        "node_features": data.x.shape[1],
        "classes": classes,
        "edge_homophily": round(float(edge_homo), 3),
        "adjusted_homophily": round(float(adj_homo), 3)
    }



In [15]:
path_to_folder = '../tmp/Cora' 
dataset = Planetoid(root=path_to_folder, name='Cora')
data = dataset[0]

stats = calculate_stats_from_data(data, "Cora")
stats

{'dataset': 'Cora',
 'nodes': 2708,
 'edges': 5278,
 'avg_degree': 3.9,
 'global_clustering': 0.093,
 'avg_local_clustering': 0.241,
 'diameter': 19,
 'node_features': 1433,
 'classes': 7,
 'edge_homophily': 0.81,
 'adjusted_homophily': 0.771}

In [18]:
path_to_folder = '../tmp/PubMed' 
dataset = Planetoid(root=path_to_folder, name='PubMed')
data = dataset[0]

stats = calculate_stats_from_data(data, "PubMed")
stats

{'dataset': 'PubMed',
 'nodes': 19717,
 'edges': 44324,
 'avg_degree': 4.5,
 'global_clustering': 0.054,
 'avg_local_clustering': 0.06,
 'diameter': 18,
 'node_features': 500,
 'classes': 3,
 'edge_homophily': 0.802,
 'adjusted_homophily': 0.686}

In [14]:
from torch_geometric.datasets import WebKB
path_to_folder = '../tmp/Cornell' 
dataset = WebKB(root=path_to_folder, name='cornell')
data = dataset[0]

stats = calculate_stats_from_data(data, "cornell")
stats

{'dataset': 'cornell',
 'nodes': 183,
 'edges': 277,
 'avg_degree': 3.03,
 'global_clustering': 0.035,
 'avg_local_clustering': 0.167,
 'diameter': 8,
 'node_features': 1703,
 'classes': 5,
 'edge_homophily': 0.123,
 'adjusted_homophily': -0.22}

In [16]:
from torch_geometric.datasets import WebKB
path_to_folder = '../tmp/Texas' 
dataset = WebKB(root=path_to_folder, name='texas')
data = dataset[0]

stats = calculate_stats_from_data(data, "texas")
stats

{'dataset': 'texas',
 'nodes': 183,
 'edges': 279,
 'avg_degree': 3.05,
 'global_clustering': 0.033,
 'avg_local_clustering': 0.198,
 'diameter': 8,
 'node_features': 1703,
 'classes': 5,
 'edge_homophily': 0.061,
 'adjusted_homophily': -0.294}

In [17]:
from torch_geometric.datasets import WebKB
path_to_folder = '../tmp/Wisconsin' 
dataset = WebKB(root=path_to_folder, name='wisconsin')
data = dataset[0]

stats = calculate_stats_from_data(data, "wisconsin")
stats

{'dataset': 'wisconsin',
 'nodes': 251,
 'edges': 450,
 'avg_degree': 3.59,
 'global_clustering': 0.039,
 'avg_local_clustering': 0.208,
 'diameter': 8,
 'node_features': 1703,
 'classes': 5,
 'edge_homophily': 0.178,
 'adjusted_homophily': -0.173}